In [1]:
import os
os.environ["MLFLOW_TRACKING_URI"]="https://dagshub.com/KishorKumarParoi/Wine-Quality-Prediction.mlflow"
os.environ["MLFLOW_TRACKING_USERNAME"]="KishorKumarParoi"
os.environ["MLFLOW_TRACKING_PASSWORD"]="c0edb772affc440233d8431ecf491d2e577ec401"

In [2]:
import os
%pwd

'/Users/kishorkumarparoi/Desktop/MLOps/Project/Wine-Quality-DS/research'

In [3]:
os.chdir("../")
%pwd

'/Users/kishorkumarparoi/Desktop/MLOps/Project/Wine-Quality-DS'

In [4]:
from dataclasses import dataclass
from pathlib import Path


@dataclass
class ModelEvaluationConfig:
    root_dir: Path
    test_data_path: Path
    model_path: Path
    all_params: dict
    metric_file_name: Path
    target_column: str
    mlflow_uri: str


In [5]:
from src.wine_quality_prediction.constants import *
from src.wine_quality_prediction.utils.common import *

In [6]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH,
        schema_filepath = SCHEMA_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        self.schema = read_yaml(schema_filepath)

        create_directories([self.config.artifacts_root])

    def get_model_evaluation_config(self) -> ModelEvaluationConfig:
        config = self.config.model_evaluation
        params = self.params.ElasticNet
        schema = self.schema.TARGET_COLUMN

        create_directories([config.root_dir])

        # Get MLflow URI from environment or use default
        mlflow_uri = os.environ.get(
            "MLFLOW_TRACKING_URI",
            "https://dagshub.com/KishorKumarParoi/Wine-Quality-Prediction.mlflow"
        )

        model_evaluation_config = ModelEvaluationConfig(
            root_dir=config.root_dir,
            test_data_path=config.test_data_path,
            model_path=config.model_path,
            all_params=params,
            metric_file_name=config.metric_file_name,
            target_column=schema.name,
            mlflow_uri=mlflow_uri
        )
        
        logger.info(f"Model evaluation config created. MLflow URI: {mlflow_uri}")
        return model_evaluation_config


In [10]:
import os
import pandas as pd
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from urllib.parse import urlparse
import mlflow
from mlflow import sklearn
import numpy as np
import joblib
from src.wine_quality_prediction import logger


In [11]:
from src.wine_quality_prediction import logger

class ModelEvaluation:
    def __init__(self, config: ModelEvaluationConfig):
        self.config = config

    def eval_metrics(self, actual, pred):
        """Calculate evaluation metrics"""
        rmse = np.sqrt(mean_squared_error(actual, pred))
        mae = mean_absolute_error(actual, pred)
        r2 = r2_score(actual, pred)
        return rmse, mae, r2
    
    def log_into_mlflow(self):
        """Log model metrics and model to MLflow"""
        try:
            test_data = pd.read_csv(self.config.test_data_path)
            model = joblib.load(self.config.model_path)

            test_x = test_data.drop([self.config.target_column], axis=1)
            test_y = test_data[[self.config.target_column]]

            mlflow.set_registry_uri(self.config.mlflow_uri)
            tracking_url_type_store = urlparse(mlflow.get_tracking_uri()).scheme

            with mlflow.start_run():
                predicted_qualities = model.predict(test_x)
                (rmse, mae, r2) = self.eval_metrics(test_y, predicted_qualities)
                
                # Save metrics locally
                scores = {"rmse": rmse, "mae": mae, "r2": r2}
                save_json(path=Path(self.config.metric_file_name), data=scores)
                logger.info(f"Metrics saved: {scores}")

                # Log parameters
                mlflow.log_params(self.config.all_params)
                logger.info(f"Parameters logged to MLflow")

                # Log metrics
                mlflow.log_metric("rmse", rmse)
                mlflow.log_metric("mae", mae)
                mlflow.log_metric("r2", r2)
                logger.info(f"Metrics logged to MLflow - RMSE: {rmse:.4f}, MAE: {mae:.4f}, R2: {r2:.4f}")

                # Try to register model, but don't fail if it doesn't work
                if tracking_url_type_store != "file":
                    try:
                        sklearn.log_model(
                            model, 
                            name="model",
                            serialization_format="skops", 
                            registered_model_name="ElasticnetModel"
                        )
                        logger.info("Model registered successfully")
                    except Exception as e:
                        logger.warning(f"Model registration failed: {str(e)}")
                        logger.info("Logging model without registration")
                        sklearn.log_model(model, "model")
                else:
                    sklearn.log_model(model, "model")
                    logger.info("Model logged to local MLflow")
                    
        except Exception as e:
            logger.error(f"MLflow logging failed: {str(e)}")
            raise e


In [12]:
try:
    config = ConfigurationManager()
    model_evaluation_config = config.get_model_evaluation_config()
    model_evaluation = ModelEvaluation(config=model_evaluation_config)
    
    logger.info("Starting model evaluation...")
    model_evaluation.log_into_mlflow()
    logger.info("Model evaluation completed successfully!")
    
except Exception as e:
    logger.error(f"Model evaluation pipeline failed: {str(e)}")
    raise e


[2026-04-17 02:35:48,043: INFO: common: yaml file: config/config.yaml loaded successfully]
[2026-04-17 02:35:48,045: INFO: common: yaml file: params.yaml loaded successfully]
[2026-04-17 02:35:48,048: INFO: common: yaml file: schema.yaml loaded successfully]
[2026-04-17 02:35:48,049: INFO: common: created directory at: artifacts]
[2026-04-17 02:35:48,049: INFO: common: created directory at: artifacts/model_evaluation]
[2026-04-17 02:35:48,049: INFO: 4100184955: Model evaluation config created. MLflow URI: https://dagshub.com/KishorKumarParoi/Wine-Quality-Prediction.mlflow]
[2026-04-17 02:35:48,050: INFO: 181115262: Starting model evaluation...]
[2026-04-17 02:35:49,169: INFO: common: json file saved at: artifacts/model_evaluation/metrics.json]
[2026-04-17 02:35:49,170: INFO: 3352595665: Metrics saved: {'rmse': np.float64(0.7064629761476984), 'mae': 0.5616829704799307, 'r2': 0.239416808423635}]
[2026-04-17 02:35:49,487: INFO: 3352595665: Parameters logged to MLflow]
[2026-04-17 02:35:50

Registered model 'ElasticnetModel' already exists. Creating a new version of this model...
2026/04/17 02:36:13 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: ElasticnetModel, version 2
Created version '2' of model 'ElasticnetModel'.


[2026-04-17 02:36:14,536: INFO: 3352595665: Model registered successfully]
🏃 View run omniscient-sow-437 at: https://dagshub.com/KishorKumarParoi/Wine-Quality-Prediction.mlflow/#/experiments/0/runs/d5ad70b746544cd38743ec3ddddf62b7
🧪 View experiment at: https://dagshub.com/KishorKumarParoi/Wine-Quality-Prediction.mlflow/#/experiments/0
[2026-04-17 02:36:15,195: INFO: 181115262: Model evaluation completed successfully!]
